# Baby Step 4 — Committee Pack, Transparently Reproduced

This notebook turns the Obsidian knowledge base into the first committee-ready product. It does **not** invent a new analytical layer. It reads the existing company universe, scenario scores, claim-level evidence and contradiction register; reproduces the decision; maps the result to a 14-slide narrative; and applies a controlled release gate to the verified PowerPoint.

**Synthetic architecture demonstration only — not investment advice.**


## What this notebook makes transparent

1. Which vault and files are being used.
2. Which companies enter the immediate review set and why.
3. Why VoltEdge is the lead case.
4. Which evidence is strong, weak or contradictory.
5. How those facts produce a narrow committee recommendation.
6. Which 14 slides communicate that recommendation.
7. Why the presentation is released—or blocked.

The default is a **dry run**. No vault file is changed unless you deliberately switch the release flag.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value):
        print(value)
    def Markdown(value):
        return value
import hashlib
import json
import os
import shutil
import sys

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DEFAULT_COLAB_VAULT = Path("/content/drive/MyDrive/Alejandro-Reynoso-Investment-Banking-Vault")
LOCAL_VAULT = Path("/workspace/scratch/9ba1ff46ede5/Alejandro-Reynoso-Investment-Banking-Vault")
VAULT = Path(os.environ.get("IB_VAULT_PATH", DEFAULT_COLAB_VAULT if RUNNING_IN_COLAB else LOCAL_VAULT))

# Safety controls. Keep both False for an inspection-only run.
RELEASE_VERIFIED_PRESENTATION = False
WRITE_REPRODUCTION_ARTIFACTS = False

PRESENTATION = VAULT / "Presentations" / "AI_Infrastructure_Strategic_Opportunities_Committee_Pack.pptx"
EXPECTED_PRESENTATION_SHA256 = "d30b4fcfd27d84724d49ad6d668ed882421ab3c72850777cd186c5b0f46755f2"

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"Vault: {VAULT}")
print(f"Dry run: {not (RELEASE_VERIFIED_PRESENTATION or WRITE_REPRODUCTION_ARTIFACTS)}")


## 1. Resolve and validate the vault


In [ ]:
required = [
    VAULT / "Data" / "company_master.csv",
    VAULT / "Data" / "loop_002_scoring_breakdown.csv",
    VAULT / "Data" / "loop_003_evidence_scores.csv",
    VAULT / "Data" / "claim_register.csv",
    VAULT / "Data" / "validation_report.json",
    PRESENTATION,
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing required files:\n" + "\n".join(missing)

validation = json.loads((VAULT / "Data" / "validation_report.json").read_text())
assert validation["companies"] == 102
assert validation["unresolved_wikilinks"] == []
assert validation["canvas_errors"] == []

inventory = {
    "companies": validation["companies"],
    "company_notes": validation["company_notes"],
    "markdown_notes": validation["markdown_notes"],
    "sectors": validation["sectors"],
    "unresolved_wikilinks": len(validation["unresolved_wikilinks"]),
    "canvas_errors": len(validation["canvas_errors"]),
}
display(pd.DataFrame([inventory]).T.rename(columns={0: "value"}))


## 2. Load the analytical layers


In [ ]:
companies = pd.read_csv(VAULT / "Data" / "company_master.csv")
scores = pd.read_csv(VAULT / "Data" / "loop_002_scoring_breakdown.csv")
evidence = pd.read_csv(VAULT / "Data" / "loop_003_evidence_scores.csv")
claims = pd.read_csv(VAULT / "Data" / "claim_register.csv")

assert len(companies) == 102 and companies["id"].is_unique
assert len(scores) == 102 and scores["id"].is_unique
assert set(evidence["id"]) == set(claims["id"])

universe = companies.merge(scores[["id", "final_score", "raw_total"]], on="id", validate="one_to_one")
print(f"Loaded {len(universe)} companies, {len(evidence)} evidence scores and {len(claims)} governed claims.")


## 3. Reproduce the immediate review set

The scenario score is a **relevance score**, not a valuation, return forecast or mandate probability. It combines declared theme/sector/keyword fit, growth, recurring revenue, moat and capital need, less execution risk.


In [ ]:
top8 = (universe.sort_values(["final_score", "name"], ascending=[False, True])
        .head(8)
        [["id", "name", "sector", "subsector", "stage", "investment_style", "transaction_type", "final_score"]]
        .reset_index(drop=True))
expected_names = {"LuminaGrid", "Beacon Fiber", "MeridianAI", "VoltEdge Thermal Systems", "OrigoCloud", "TensorPeak", "Futura Towers", "Halo Broadband"}
assert set(top8["name"]) == expected_names
display(top8)

ax = top8.sort_values("final_score").plot.barh(x="name", y="final_score", figsize=(10, 5), legend=False, color="#3D8DFF")
ax.set_xlim(0, 100); ax.set_xlabel("Scenario relevance / 100"); ax.set_ylabel("")
ax.set_title("Immediate review set")
plt.tight_layout(); plt.show()


## 4. Map the review set to mandate families


In [ ]:
portfolio = pd.DataFrame([
    ["OPP-007", "LuminaGrid", "Growth capital", 95],
    ["OPP-008", "Beacon Fiber", "Strategic M&A", 95],
    ["OPP-009", "MeridianAI", "Strategic M&A", 95],
    ["OPP-010", "VoltEdge Thermal Systems", "Growth capital", 95],
    ["OPP-011", "OrigoCloud", "Strategic M&A", 92],
    ["OPP-012", "TensorPeak", "Growth capital", 92],
    ["OPP-013", "Ionic Energy", "Infrastructure financing", int(universe.loc[universe.name.eq("Ionic Energy"), "final_score"].iloc[0])],
    ["OPP-014", "Distrito Data Centers", "Infrastructure financing", int(universe.loc[universe.name.eq("Distrito Data Centers"), "final_score"].iloc[0])],
], columns=["opportunity_id", "company", "mandate_family", "score"])

assert portfolio["opportunity_id"].is_unique
display(portfolio)
display(portfolio.groupby("mandate_family").agg(opportunities=("opportunity_id", "count"), average_score=("score", "mean")).round(1))


## 5. Reproduce the VoltEdge lead case


In [ ]:
volt = universe.loc[universe["name"].eq("VoltEdge Thermal Systems")].iloc[0]
volt_snapshot = pd.Series({
    "FY2024 revenue ($m)": volt["revenue_prev_usd_m"],
    "FY2025 revenue ($m)": volt["revenue_usd_m"],
    "Revenue growth": f"{volt['revenue_growth_pct']:.1f}%",
    "EBITDA ($m)": volt["ebitda_usd_m"],
    "EBITDA margin": f"{volt['ebitda_margin_pct']:.1f}%",
    "Free cash flow ($m)": volt["free_cash_flow_usd_m"],
    "Enterprise value ($m)": volt["enterprise_value_usd_m"],
    "Recurring revenue": f"{volt['recurring_revenue_pct']:.0f}%",
    "Top-customer concentration": f"{volt['top_customer_concentration_pct']:.0f}%",
    "Scenario relevance": f"{volt['final_score']:.0f}/100",
})
assert volt["revenue_prev_usd_m"] == 105.0
assert volt["revenue_usd_m"] == 147.0
assert volt["free_cash_flow_usd_m"] == -6.5
assert volt["final_score"] == 95
display(volt_snapshot.to_frame("value"))


## 6. Apply the evidence and contradiction layer

This is where the vault begins to exercise judgment. A high opportunity score can prioritize a company, but weak or contradictory evidence can narrow the authorized action.


In [ ]:
claim_view = claims[["id", "statement", "claim_type", "sources", "confidence_score", "status", "stale"]].copy()
display(claim_view)

average_confidence = float(evidence["confidence_score"].mean())
capacity_claim = claims.loc[claims["id"].eq("CLM-004")].iloc[0]
has_capacity_contradiction = (
    "78%" in capacity_claim["statement"]
    and "92%" in capacity_claim["statement"]
    and capacity_claim["status"] in {"Contradicted", "Contradiction open"}
)
assert round(average_confidence, 1) == 64.5
assert has_capacity_contradiction

print(f"Average confidence: {average_confidence:.1f}/100")
print(f"Capacity contradiction preserved: {has_capacity_contradiction}")

ax = evidence.plot.barh(x="id", y="confidence_score", figsize=(9, 4), legend=False, color="#6DCBF4")
ax.set_xlim(0, 100); ax.set_xlabel("Claim confidence / 100"); ax.set_ylabel("")
ax.axvline(70, color="#C83E4D", linestyle="--", linewidth=1)
plt.tight_layout(); plt.show()


## 7. Run the committee decision engine


In [ ]:
conditions = {
    "universe_valid": len(universe) == 102 and not validation["unresolved_wikilinks"],
    "lead_case_relevant": int(volt["final_score"]) >= 90,
    "evidence_usable": average_confidence >= 60,
    "valuation_blocked": has_capacity_contradiction,
    "external_outreach_blocked": has_capacity_contradiction,
}

if all([conditions["universe_valid"], conditions["lead_case_relevant"], conditions["evidence_usable"]]):
    recommendation = "APPROVE INTERNAL PRELIMINARY DILIGENCE"
else:
    recommendation = "HOLD"

guardrails = [
    "Preserve the 78%–92% contradiction",
    "No external outreach",
    "No valuation reliance",
    "No mandate approval",
    "Return with reconciled evidence and a refreshed recommendation",
]

display(pd.DataFrame([conditions]).T.rename(columns={0: "result"}))
display(Markdown(f"### {recommendation}\n\n" + "\n".join(f"- {g}" for g in guardrails)))
assert recommendation == "APPROVE INTERNAL PRELIMINARY DILIGENCE"


## 8. Build the 14-slide communication manifest


In [ ]:
slides = [
    (1, "AI Infrastructure Strategic Opportunities", "Define the committee product and synthetic scope"),
    (2, "Authorize diligence—not outreach", "Lead with the narrow recommendation and guardrails"),
    (3, "The opportunity graph changed materially", "Show why the scenario requires a fresh screen"),
    (4, "Eight companies define the immediate review set", "Present the top-ranked review universe"),
    (5, "Eight opportunities create three mandate families", "Translate companies into actionable banking lenses"),
    (6, "The opportunity is an ecosystem—not a single subsector", "Show the cross-sector origination logic"),
    (7, "VoltEdge combines strategic relevance with capital need", "Explain the 95-point prioritization"),
    (8, "Growth is strong; cash generation is not", "Frame financial momentum and funding need"),
    (9, "A disciplined growth-capital process is the leading lens", "State the preliminary transaction thesis"),
    (10, "Evidence is usable—but uneven", "Expose confidence rather than hide it"),
    (11, "One contradiction blocks valuation reliance", "Preserve the 92% versus 78% conflict"),
    (12, "Four workstreams can resolve the decision", "Convert uncertainty into a diligence program"),
    (13, "The committee has three defensible choices", "Make approve, hold and reject explicit"),
    (14, "Decision requested: authorize the controlled next step", "Close with the exact authorization requested"),
]
slide_manifest = pd.DataFrame(slides, columns=["slide", "title", "communication_job"])
assert len(slide_manifest) == 14 and slide_manifest["slide"].is_unique
display(slide_manifest)


## 9. Verify the editable PowerPoint before any release

The PowerPoint is the governed visual layer. This notebook recalculates its inputs and checks the approved master’s SHA-256 fingerprint. If the deck changes, the release gate blocks until the new version is reviewed and its fingerprint deliberately updated.


In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

actual_hash = sha256(PRESENTATION)
hash_matches = actual_hash == EXPECTED_PRESENTATION_SHA256
release_checks = {
    **conditions,
    "recommendation_reproduced": recommendation == "APPROVE INTERNAL PRELIMINARY DILIGENCE",
    "slide_manifest_complete": len(slide_manifest) == 14,
    "verified_presentation_hash": hash_matches,
}
display(pd.DataFrame([release_checks]).T.rename(columns={0: "passed"}))
print("Presentation SHA-256:", actual_hash)
assert all(release_checks.values()), "Release blocked: one or more analytical or presentation checks failed."


## 10. Optional governed write and release


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = VAULT / "Reports" / "Committee Pack Reproductions" / run_stamp

manifest_payload = {
    "generated_at_utc": run_stamp,
    "vault": str(VAULT),
    "recommendation": recommendation,
    "guardrails": guardrails,
    "average_confidence": round(average_confidence, 1),
    "capacity_contradiction": has_capacity_contradiction,
    "presentation_sha256": actual_hash,
    "release_checks": release_checks,
    "slides": slide_manifest.to_dict(orient="records"),
}

if WRITE_REPRODUCTION_ARTIFACTS or RELEASE_VERIFIED_PRESENTATION:
    assert all(release_checks.values())
    output_dir.mkdir(parents=True, exist_ok=False)

if WRITE_REPRODUCTION_ARTIFACTS:
    (output_dir / "committee_pack_manifest.json").write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
    top8.to_csv(output_dir / "immediate_review_set.csv", index=False)
    portfolio.to_csv(output_dir / "opportunity_portfolio.csv", index=False)

if RELEASE_VERIFIED_PRESENTATION:
    released = output_dir / f"AI_Infrastructure_Strategic_Opportunities_Committee_Pack_{run_stamp}.pptx"
    shutil.copy2(PRESENTATION, released)
    assert sha256(released) == EXPECTED_PRESENTATION_SHA256
    print("Released verified presentation:", released)
else:
    print("DRY RUN — verified presentation was not copied and the vault was not changed.")


## 11. Interpretation of Baby Step 4

The vault is no longer only a repository. It now performs a controlled conversion:

**Universe → scenario relevance → opportunity set → evidence quality → contradiction → recommendation → committee communication → authorized next work.**

The most important judgment is not “VoltEdge scores 95.” It is: **the 95 score authorizes attention, while the 78%–92% contradiction restricts what the committee may authorize.** That separation between attractiveness and evidentiary permission is the operating-system behavior we wanted to demonstrate.
